In [ ]:
from openai import OpenAI
import json 
import os 
from dotenv import load_dotenv
from IPython.display import Markdown

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
openAI = OpenAI()

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama_url="http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
alex_model = "gpt-5.4-mini-2026-03-17"
blake_model = "llama3.2"
charlie_model = "gpt-4o-mini"

In [ ]:
alex_prompt = """
Philosophy: Vacation = Pampering, Status, and Comfort.
Top Pick: The Maldives or Monaco.
Traits: Prefers 5-star resorts, private villas, and "Instagrammable" aesthetics. Values convenience and exclusivity above all.
Tone: Sophisticated, slightly elitist, dismisses anything "budget" or "exhausting."
"""

blake_prompt = """
Philosophy: Vacation = Movement, Challenge, and Exploration.
Top Pick: South Africa (Safaris) or Dahab (Diving/Hiking).
Traits: Hates staying in one place. Wants to hike, dive, or skydive. Prefers hostels or camps over fancy hotels.
Tone: Energetic, blunt, finds Alex’s style "boring" and "lazy."
"""

charlie_prompt = """
Philosophy: Vacation = History, Soul, and Quiet Beauty.
Top Pick: Santorini (Greece) or Tuscany (Italy).
Traits: Loves walking through old streets, visiting museums, and local cafes. Values "The Vibe" and authenticity.
Tone: Calm, poetic, intellectual. Acts as the mediator but remains stubborn about their love for "culture" over "resorts" or "extreme sports."
"""


In [ ]:
def convert_to_json (chat) : 
  return json.dumps (chat, ensure_ascii= False , indent= 2)

In [ ]:
def build_user_prompt (agent_name , history_chat ) : 
  chat_to_json = convert_to_json (history_chat)
  user_prompt = f"""
  you are {agent_name} in conversation with the two other participants .
  this conversation is follow as : 
  {chat_to_json}

  now respond as {agent_name} 
  Try to reply with a maximum of 150 words to sound more human.
  """
  return user_prompt

In [ ]:

def call_agent (model , system_prompt , user_prompt , client) : 
  responses = client.chat.completions.create ( 
    model = model   ,
    messages = [ 
      {"role" : "system" , "content" : system_prompt} , 
      {"role" : "user" , "content" : user_prompt}
    ],
  )

  return responses.choices[0].message.content

In [ ]:



agent_chat = [ 
  {"speaker" : "Alex" , "content" : "What is your destination for vacation?"}
]
agent_info = [ 
  {"name" :"blake" , "model_name" :blake_model , "system_prompt" : blake_prompt , "client" : ollama },
  {"name" :"charlie" , "model_name" :charlie_model , "system_prompt" : charlie_prompt , "client" : openAI},
  {"name" :"Alex" , "model_name" :alex_model , "system_prompt" : alex_prompt , "client" :openAI }
]

for _ in range (4) : 
  for agent in agent_info : 
    User = build_user_prompt(agent["name"] ,agent_chat)
    response = call_agent (agent["model_name"],agent["system_prompt"] , User ,agent["client"])
    agent_chat.append ({"speaker" : agent["name"] , "content" : response })
    display(Markdown (f"### {agent["name"]} :\n{response} "))

In [ ]:
display (agent_chat)